# 02 — Fine-tuning DistilBERT pour la classification spam

Ce notebook couvre :
1. Chargement du dataset construit dans 01_explore
2. Tokenisation avec DistilBERT
3. Fine-tuning avec HuggingFace Trainer
4. Évaluation et comparaison avec le baseline SVM
5. Sauvegarde du modèle fine-tuné

## 0. Installation

In [ ]:
# !pip install transformers datasets torch accelerate scikit-learn pandas

## 1. Chargement du dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('dataset.csv')
df['text_clean'] = df['text_clean'].fillna('')

X = df['text_clean'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train : {len(X_train)} | Test : {len(X_test)}')
print(f'Répartition train — Ham: {(y_train==0).sum()} | Spam: {(y_train==1).sum()}')

## 2. Tokenisation DistilBERT

In [ ]:
from transformers import DistilBertTokenizerFast
import torch
from torch.utils.data import Dataset

MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN    = 256  # On tronque à 256 tokens (emails souvent longs)

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class EmailDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding='max_length',
            max_length=max_len,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx]
        }

train_dataset = EmailDataset(X_train, y_train, tokenizer, MAX_LEN)
test_dataset  = EmailDataset(X_test,  y_test,  tokenizer, MAX_LEN)

print(f'✓ Datasets tokenisés — Train: {len(train_dataset)} | Test: {len(test_dataset)}')

## 3. Chargement du modèle pré-entraîné

On utilise `DistilBertForSequenceClassification` avec 2 classes (ham / spam).
Seule la tête de classification est initialisée aléatoirement — le reste des poids vient du pré-entraînement.

In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'ham', 1: 'spam'},
    label2id={'ham': 0, 'spam': 1}
)

# Paramètres entraînables vs gelés
total  = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Paramètres totaux    : {total:,}')
print(f'Paramètres entraînables : {trainable:,)')

## 4. Configuration de l'entraînement

> **💡 Astuce** : Si tu as peu de données (<200 exemples), gèle les couches
> de l'encodeur et entraîne uniquement la tête de classification :
> ```python
> for param in model.distilbert.parameters():
>     param.requires_grad = False
> ```
> Avec >500 exemples, le fine-tuning complet fonctionne mieux.

In [ ]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':  accuracy_score(labels, preds),
        'f1':        f1_score(labels, preds, average='binary'),
        'precision': precision_score(labels, preds, average='binary'),
        'recall':    recall_score(labels, preds, average='binary'),
    }

training_args = TrainingArguments(
    output_dir='./distilbert_spam',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=2e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_dir='./logs',
    logging_steps=10,
    fp16=torch.cuda.is_available(),  # Mixed precision si GPU disponible
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print('✓ Trainer configuré')

## 5. Entraînement

In [ ]:
trainer.train()
results = trainer.evaluate()
print('\n=== Résultats finaux ===')
for k, v in results.items():
    print(f'  {k}: {v:.4f}')

## 6. Courbes d'apprentissage

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_loss = [(e['epoch'], e['loss']) for e in log_history if 'loss' in e and 'eval_loss' not in e]
eval_f1    = [(e['epoch'], e['eval_f1']) for e in log_history if 'eval_f1' in e]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

if train_loss:
    epochs_loss, losses = zip(*train_loss)
    ax1.plot(epochs_loss, losses, color='steelblue')
    ax1.set_title('Loss d\'entraînement')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')

if eval_f1:
    epochs_f1, f1s = zip(*eval_f1)
    ax2.plot(epochs_f1, f1s, color='tomato', marker='o')
    ax2.set_title('F1 score (validation)')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('F1')
    ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 7. Matrice de confusion — DistilBERT

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

preds_output = trainer.predict(test_dataset)
y_pred = np.argmax(preds_output.predictions, axis=-1)

print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.title('Matrice de confusion — DistilBERT fine-tuné')
plt.tight_layout()
plt.savefig('confusion_matrix_distilbert.png', dpi=150)
plt.show()

## 8. Sauvegarde du modèle

In [ ]:
model.save_pretrained('./distilbert_spam_final')
tokenizer.save_pretrained('./distilbert_spam_final')
print('✓ Modèle fine-tuné sauvegardé dans ./distilbert_spam_final')
print('   → Utilise ce chemin dans pipeline.py pour l\'inférence automatique')

## 9. Inférence rapide — test sur un mail custom

In [ ]:
from transformers import pipeline as hf_pipeline

classifier = hf_pipeline(
    'text-classification',
    model='./distilbert_spam_final',
    tokenizer='./distilbert_spam_final',
    truncation=True,
    max_length=256
)

test_emails = [
    "Congratulations! You won $1,000,000. Click here to claim your prize now!",
    "Hi, just checking in about our meeting tomorrow at 10am. See you then!",
    "URGENT: Your account has been compromised. Verify immediately."
]

for email in test_emails:
    result = classifier(email)[0]
    print(f"[{result['label'].upper()} — {result['score']:.2%}] {email[:60]}...")